# AutoInspect AI: Deep Learning Vehicle Damage Classifier & Grad-CAM Pipeline
### End-to-End Transfer Learning on ResNet50 for Insurance Claim Automation

This notebook documents the full machine learning lifecycle:
1. **Dataset Loading & Augmentation** (ImageNet standardization, Color Jitter, Flips)
2. **Model Architecture** (ResNet50 Backbone + Dual Head Classifier)
3. **Training & Validation Loop** (Cosine Annealing LR, CrossEntropy Loss, Early Checkpointing)
4. **Evaluation Metrics** (Accuracy, Macro F1, Confusion Matrix)
5. **Visual Explainability (Grad-CAM)** (Gradient backpropagation on Layer 4 activations)

In [ ]:
import os
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Computation Device: {device}')

## 1. Dataset Generation & Loading
We load train, validation, and test splits with data augmentation to prevent overfitting on specific lighting conditions or vehicle angles.

In [ ]:
from dataset_loader import create_dataloaders
from generate_synthetic_data import generate_dataset, DATA_DIR

# Verify / generate data structure
if not (DATA_DIR / 'train').exists():
    generate_dataset(DATA_DIR, samples_per_class_train=40, samples_per_class_val=10)

train_loader, val_loader, test_loader, class_names = create_dataloaders(str(DATA_DIR), batch_size=16)
print(f'Damage Categories: {class_names}')
print(f'Train Samples: {len(train_loader.dataset)} | Val Samples: {len(val_loader.dataset)}')

## 2. Model Architecture: ResNet50 Transfer Learning
We leverage a pretrained **ResNet50** convolutional backbone, freezing early feature extractors and fine-tuning higher-level representations for fine-grained damage texture analysis.

In [ ]:
class DamageClassifierNet(nn.Module):
    def __init__(self, num_damage_classes=5, num_severity_classes=4, pretrained=True):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet50(weights=weights)
        in_features = self.backbone.fc.in_features  # 2048
        self.backbone.fc = nn.Identity()

        self.shared_fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35)
        )

        self.damage_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_damage_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        shared = self.shared_fc(features)
        return self.damage_head(shared)

model = DamageClassifierNet(num_damage_classes=len(class_names)).to(device)
print(model)

## 3. Training & Validation Execution

In [ ]:
from train import train_model

# Execute training pipeline
history = train_model(data_dir=str(DATA_DIR), epochs=8, batch_size=16, learning_rate=1e-4)

## 4. Evaluation & Confusion Matrix

In [ ]:
from evaluate import evaluate_model

# Run test evaluation and export confusion matrix
evaluate_model(data_dir=str(DATA_DIR))

## 5. Grad-CAM Visual Explainability
Visualizing the class activation map demonstrates that the model focuses on the damage region rather than the background or vehicle wheels.

In [ ]:
sys.path.append(str(Path.cwd().parent / 'backend'))
from app.ml.classifier import get_predictor

predictor = get_predictor()
test_img_path = list((DATA_DIR / 'test' / 'scratch').glob('*.jpg'))[0]
pil_test = Image.open(test_img_path)

result = predictor.predict(pil_test)
print(f"Predicted Class: {result['damage_display_name']}")
print(f"Severity: {result['severity']} | Confidence: {result['confidence'] * 100:.1f}%")
print(f"Estimated Repair Cost: ${result['estimated_cost']['min']} - ${result['estimated_cost']['max']}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(pil_test)
ax1.set_title('Original Vehicle Image')
ax1.axis('off')

ax2.imshow(result['gradcam_pil'])
ax2.set_title('Grad-CAM Heatmap Localization')
ax2.axis('off')
plt.show()